# DIVE-HMM · CCBF — Notebook 2 (Headline experiments)

This notebook runs the **paper's headline evaluation** for Calibrated Condition-Belief Filtering (CCBF):

1. **Multi-condition localization** — CCBF vs single-image, fixed-observation filter, SeqSLAM-style, and an
   uncertainty baseline (SUE / L2), on genuinely multi-condition data.
2. **Calibration** — ECE and **reliability diagrams** (the headline reliability metric).
3. **Open-set / off-map** — **AUROC / AUPR** for "lost" detection.
4. **Theory validation** — the **ECE-vs-sequence-length (T)** curve that directly tests *Theorem 1
   (miscalibration propagation)*: a fixed, condition-agnostic observation model should degrade with T while CCBF
   stays flat.
5. **Label-free test-time domain adaptation** — condition belief vs. true condition, and condition-drift robustness.

**Datasets.** The robust backbone is **Nordland Full** (4 synchronized seasons, clean frame-ID GT → ideal,
*controlled* multi-condition testbed for calibration/open-set; note these metrics are **not** saturated the way
recall is). **RobotCar Seasons v2** and **Extended CMU** are added as hard real-world conditions via a *best-effort,
self-probing* loader — if the pose-GT format matches the standard schema it runs; otherwise it prints the format so
the loader can be finalized in NB3.

> Run top-to-bottom. `CONFIG.SMOKE_TEST=True` (default) uses a small subset so the full suite finishes in minutes.
> Offline / Kaggle Internet OFF. Reuses the descriptor cache in `/kaggle/working/dive_hmm/features` if present.

In [ ]:
# === P0.0 — Imports, device, seeds (same contract as NB1) ==============================
import os, sys, json, time, glob, math, warnings
from dataclasses import dataclass, field, asdict
from collections import OrderedDict, defaultdict
import numpy as np

REPORT = OrderedDict()
def log(step, status, msg=""):
    REPORT[step] = {"status": status, "msg": msg}
    print({"ok": "[ OK ]", "warn": "[WARN]", "fail": "[FAIL]", "info": "[info]"}.get(status, "[??]"), step + ":", msg)

def _seed(s=0):
    import random; random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except Exception: pass

try:
    import torch; _HAS_TORCH = True
except Exception as e:
    _HAS_TORCH = False; print("torch not importable:", e)
_seed(0)
if _HAS_TORCH:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BF16_OK = (DEVICE == "cuda" and torch.cuda.is_bf16_supported())
    print("torch", torch.__version__, "| device", DEVICE, "| bf16", BF16_OK)
    if DEVICE == "cuda": print("GPU:", torch.cuda.get_device_properties(0).name)
else:
    DEVICE, BF16_OK = "cpu", False
print("numpy", np.__version__)

In [ ]:
# === P0.1 — CONFIG =====================================================================
@dataclass
class Config:
    SMOKE_TEST: bool = True
    SEED: int = 0
    INPUT_ROOTS: tuple = ("/kaggle/input",)
    WORKING_DIR: str = "/kaggle/working/dive_hmm"
    CACHE_SUBDIR: str = "features"; CKPT_SUBDIR: str = "ckpt"

    IMG_SIZE: int = 322
    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 2
    USE_BF16: bool = True
    SMOKE_MAX_FRAMES: int = 3000          # per season/condition in smoke

    # CCBF
    STICKY_RHO: float = 0.97
    MOTION_STD: float = 2.0
    MOTION_VELOCITY: float = 1.0
    MOTION_LEAK: float = 1e-3
    NULL_PRIOR: float = 0.02
    NULL_SELF: float = 0.9

    # evaluation
    RECALL_KS: tuple = (1, 5, 10)
    PLACE_TOL: int = 2
    ECE_BINS: int = 15
    USE_TRAIN_CONDITION_LABELS: bool = True

    # experiments
    T_GRID: tuple = (1, 2, 3, 5, 8, 13, 21, 34, 55, 89)   # sequence lengths for ECE-vs-T
    OFFMAP_MAP_FRACTION: float = 0.7      # first 70% of the route is the map; last 30% are off-map queries
    ROBOTCAR_REF_COND: str = "overcast-reference"
    ROBOTCAR_QUERY_CONDS: tuple = ("night", "night-rain", "snow", "rain", "dusk", "sun")
    POSE_TOL_M: float = 5.0               # metres: a retrieval is correct if within this of GT pose

    @property
    def cache_dir(self): return os.path.join(self.WORKING_DIR, self.CACHE_SUBDIR)
    @property
    def ckpt_dir(self): return os.path.join(self.WORKING_DIR, self.CKPT_SUBDIR)

CONFIG = Config()
if _HAS_TORCH and DEVICE == "cuda":
    g = torch.cuda.get_device_properties(0).total_memory / 1e9
    CONFIG.BATCH_SIZE = 256 if g >= 70 else (128 if g >= 40 else 64)
CONFIG.USE_BF16 = CONFIG.USE_BF16 and BF16_OK
os.makedirs(CONFIG.cache_dir, exist_ok=True); os.makedirs(CONFIG.ckpt_dir, exist_ok=True)
print(json.dumps({k: str(v) for k, v in asdict(CONFIG).items()}, indent=0)[:800])
log("P0.1.config", "ok", f"SMOKE={CONFIG.SMOKE_TEST} batch={CONFIG.BATCH_SIZE} bf16={CONFIG.USE_BF16}")

In [ ]:
# === P0.2 — Offline env + path discovery ==============================================
for k, v in {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1",
             "PIP_NO_INDEX": "1", "WANDB_MODE": "offline", "TOKENIZERS_PARALLELISM": "false"}.items():
    os.environ.setdefault(k, v)

def _find_dir(markers_any, markers_all=(), roots=None, max_depth=4):
    roots = roots or CONFIG.INPUT_ROOTS; seen = set()
    for root in roots:
        if not os.path.isdir(root): continue
        frontier = [(root, 0)]
        while frontier:
            d, dep = frontier.pop(0)
            if d in seen or dep > max_depth: continue
            seen.add(d)
            try: entries = set(os.listdir(d))
            except Exception: continue
            ok_all = all(any(m.lower() in e.lower() for e in entries) for m in markers_all) if markers_all else True
            ok_any = any(any(m.lower() in e.lower() for e in entries) for m in markers_any) if markers_any else True
            if ok_all and ok_any: return d
            for e in sorted(entries):
                sub = os.path.join(d, e)
                if os.path.isdir(sub): frontier.append((sub, dep + 1))
    return None

PATHS = {
    "bundle":        _find_dir(["checkpoints"], ["checkpoints", "sources"]),
    "nordland_full": _find_dir(["data"], ["data", "annotations"]),
    "nordland_short":_find_dir(["summer", "winter"], ["summer", "winter"]),
    "robotcar":      _find_dir(["images"], ["images", "3D-models"]),
    "cmu":           _find_dir(["slice2", "slice6", "slice7"]),
}
for n, p in PATHS.items():
    log(f"P0.2.path.{n}", "ok" if p else "warn", p or "not found")
if PATHS["bundle"]:
    for sub in ["salad", "dinov2", "Hierarchical-Localization", "LightGlue", ""]:
        p = os.path.join(PATHS["bundle"], "sources", sub) if sub else os.path.join(PATHS["bundle"], "sources")
        if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p)

In [ ]:
# === P0.3 — Pluggable feature extractor (SALAD -> DINOv2 CLS+mean -> placeholder) ======
# NOTE: DINOv2 tokens are SIGNED; we use CLS + mean pooling (the GeM-clamp bug from NB1 v1 is fixed here).
EXTRACTOR_INFO = {"name": None, "dim": None, "detail": ""}

def _try_salad():
    """serizba/salad via helper.get_* (released dino_salad.ckpt configs), bypassing VPRModel. dim=8448."""
    if not PATHS["bundle"]: return None
    ck = os.path.join(PATHS["bundle"], "checkpoints", "dino_salad.ckpt")
    salad_dir = os.path.join(PATHS["bundle"], "sources", "salad")
    if not (os.path.isfile(ck) and os.path.isdir(salad_dir)): return None
    try:
        if salad_dir not in sys.path: sys.path.insert(0, salad_dir)
        from models import helper
        backbone = helper.get_backbone("dinov2_vitb14",
            {"num_trainable_blocks": 4, "return_token": True, "norm_layer": True})
        aggregator = helper.get_aggregator("SALAD",
            {"num_channels": 768, "num_clusters": 64, "cluster_dim": 128, "token_dim": 256})
        sd = torch.load(ck, map_location="cpu"); sd = sd.get("state_dict", sd)
        bb = {k[len("backbone."):]: v for k, v in sd.items() if k.startswith("backbone.")}
        ag = {k[len("aggregator."):]: v for k, v in sd.items() if k.startswith("aggregator.")}
        backbone.load_state_dict(bb, strict=False); aggregator.load_state_dict(ag, strict=False)
        model = torch.nn.Module(); model.backbone = backbone; model.aggregator = aggregator
        model.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x): return model.aggregator(model.backbone(x))
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all() or d.ndim != 2 or d.shape[1] < 1000: return None
        return fwd, d.shape[1]
    except Exception:
        import traceback; print("  SALAD error:"); traceback.print_exc(); return None

def _try_dinov2():
    pth = os.path.join(PATHS["bundle"], "checkpoints", "dinov2_vitb14_pretrain.pth") if PATHS["bundle"] else None
    try:
        vit_base = __import__("dinov2.models.vision_transformer", fromlist=["vit_base"]).vit_base
        vit = vit_base(patch_size=14, img_size=518, init_values=1.0, block_chunks=0)
    except Exception as e:
        print("  DINOv2 build failed:", repr(e)[:120]); return None
    try:
        if pth and os.path.isfile(pth):
            sd = torch.load(pth, map_location="cpu"); sd = sd.get("state_dict", sd)
            m, u = vit.load_state_dict(sd, strict=False); print(f"  DINOv2 weights (missing={len(m)})")
        vit.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x):
            f = vit.forward_features(x)
            if isinstance(f, dict):
                parts = []
                if f.get("x_norm_clstoken") is not None: parts.append(f["x_norm_clstoken"])
                if f.get("x_norm_patchtokens") is not None: parts.append(f["x_norm_patchtokens"].mean(1))
                return torch.cat(parts, 1) if parts else f
            return f.mean(1) if f.dim() == 3 else f
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all(): return None
        return fwd, d.shape[1]
    except Exception as e:
        print("  DINOv2 error:", repr(e)[:120]); return None

def _placeholder():
    GRID = 4
    @torch.no_grad()
    def fwd(x):
        B, C, H, W = x.shape; gh, gw = H // GRID, W // GRID; feats = []
        for i in range(GRID):
            for j in range(GRID):
                t = x[:, :, i*gh:(i+1)*gh, j*gw:(j+1)*gw]
                feats += [t.mean((2, 3)), t.std((2, 3)), (t[:, :, :, 1:] - t[:, :, :, :-1]).abs().mean((2, 3))]
        return torch.cat(feats, 1)
    return fwd, GRID * GRID * 9

def build_extractor():
    for name, b in [("salad", _try_salad), ("dinov2", _try_dinov2)]:
        if PATHS["bundle"]:
            print("try extractor:", name); r = b()
            if r is not None:
                EXTRACTOR_INFO.update(name=name, dim=r[1], detail="bundle")
                log("P0.3.extractor", "ok", f"{name} dim={r[1]}"); return r[0]
    fwd, dim = _placeholder(); EXTRACTOR_INFO.update(name="placeholder", dim=dim, detail="DEGRADED")
    log("P0.3.extractor", "warn", f"placeholder dim={dim} (not publication-grade)"); return fwd

EXTRACT_FWD = build_extractor()

## Shared library — metrics, baselines, and the CCBF core

The place/condition filter, calibrated observation model, and metrics are the **validated core from NB1**
(same equations as `CCBF_methodology.md`). We add three evaluation tools used by the headline experiments:
a **reliability diagram**, **AUROC** for open-set detection, and the **SUE / L2** uncertainty baseline
(Zaffar et al., CVPR 2024: the L2 distance to the nearest reference is a strong training-free uncertainty).

In [ ]:
# === L.1 — Metrics =====================================================================
def cosine_sim(Q, R):
    Q = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    R = R / (np.linalg.norm(R, axis=1, keepdims=True) + 1e-12)
    return Q @ R.T

def _logsumexp(x, axis=None):
    x = np.asarray(x, dtype=np.float64)
    m = np.max(x, axis=axis, keepdims=True)
    s = m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))
    return float(s.reshape(())) if axis is None else np.squeeze(s, axis=axis)

def recall_at_k(sim, q_ids, r_ids, ks=(1, 5, 10), tol=2):
    r_ids = np.asarray(r_ids); q_ids = np.asarray(q_ids)
    kmax = min(max(ks), sim.shape[1]); topk = np.argsort(-sim, axis=1)[:, :kmax]; out = {}
    for k in ks:
        kk = min(k, sim.shape[1])
        out[k] = float((np.abs(r_ids[topk[:, :kk]] - q_ids[:, None]) <= tol).any(1).mean())
    return out

def ece_from_conf(conf, correct, n_bins=15):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    bins = np.linspace(0, 1, n_bins + 1); ece, rows, N = 0.0, [], len(conf)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0: continue
        c, a, w = conf[m].mean(), correct[m].mean(), m.sum() / N
        ece += w * abs(a - c); rows.append((0.5 * (lo + hi), c, a, int(m.sum())))
    return float(ece), rows

def pred_correct(pred_idx, q_ids, r_ids, tol=2):
    return (np.abs(np.asarray(r_ids)[pred_idx] - np.asarray(q_ids)) <= tol).astype(float)

def fit_temperature(sim, gt_idx, grid=None):
    grid = grid if grid is not None else np.logspace(-2.2, 0.3, 30)
    v = gt_idx >= 0; s, g = sim[v], gt_idx[v]
    best, bn = grid[0], np.inf
    for tau in grid:
        lp = s / tau; lp = lp - _logsumexp(lp, axis=1)[:, None]
        nll = -lp[np.arange(len(g)), g].mean()
        if nll < bn: bn, best = nll, tau
    return float(best), float(bn)

def auroc(scores, labels):
    """Mann-Whitney AUROC; higher score => predicted positive. labels in {0,1}."""
    scores = np.asarray(scores, float); labels = np.asarray(labels, int)
    P, N = int(labels.sum()), int((labels == 0).sum())
    if P == 0 or N == 0: return float("nan")
    ranks = np.argsort(np.argsort(scores)) + 1               # ascending ranks, ties approx
    return float((ranks[labels == 1].sum() - P * (P + 1) / 2) / (P * N))

def average_precision(scores, labels):
    order = np.argsort(-np.asarray(scores)); y = np.asarray(labels, int)[order]
    tp = np.cumsum(y); prec = tp / (np.arange(len(y)) + 1); rec = tp / max(1, y.sum())
    drec = np.diff(np.concatenate([[0], rec])); return float((prec * drec).sum())

def sue_uncertainty(sim):
    """SUE/L2 (Zaffar'24): L2 distance to nearest reference = sqrt(2 - 2*max cos). Higher => less certain."""
    return np.sqrt(np.maximum(2 - 2 * sim.max(1), 0.0))

In [ ]:
# === L.2 — Motion model + fixed-observation Bayes filter (baseline) ====================
def make_motion_kernel(velocity, std):
    lo, hi = int(math.floor(velocity - 4 * std)), int(math.ceil(velocity + 4 * std))
    offs = np.arange(lo, hi + 1); w = np.exp(-0.5 * ((offs - velocity) / max(std, 1e-3)) ** 2)
    return offs, w / w.sum()

def predict_place(place, offs, w, leak):
    M = place.shape[0]; new = np.zeros_like(place)
    for off, wt in zip(offs, w):
        if off == 0: new += wt * place
        elif off > 0:
            if off < M: new[off:] += wt * place[:M - off]
        else:
            a = -off
            if a < M: new[:M - a] += wt * place[a:]
    s = new.sum()
    if s > 0: new /= s
    return (1 - leak) * new + leak / M

def run_place_filter(sim, tau, null_quantile=0.98, exit_prob=0.01):
    T, M = sim.shape; offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)
    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    for t in range(T):
        pp = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) + null * (1 - CONFIG.NULL_SELF) / M
        pn = CONFIG.NULL_SELF * null + exit_prob * place.sum()
        logb = np.empty(M + 1)
        logb[:M] = np.log(pp + 1e-30) + sim[t] / tau
        logb[M] = np.log(pn + 1e-30) + np.quantile(sim[t], null_quantile) / tau
        logb -= _logsumexp(logb); b = np.exp(logb)
        place, null = b[:M], b[M]; s = place.sum() + null; place, null = place / s, null / s
        preds[t], confs[t], nulls[t] = int(np.argmax(place)), float(place.max()), float(null)
    return preds, confs, nulls

In [ ]:
# === L.3 — Condition model + CCBF (validated core from NB1) ============================
def _pairwise_sq_dists(X, C):
    return np.maximum((X * X).sum(1)[:, None] + (C * C).sum(1)[None, :] - 2.0 * (X @ C.T), 0.0)

def cond_log_resp(desc, COND):
    logr = -0.5 * _pairwise_sq_dists(desc.astype(np.float32), COND["C"]) / (COND["sigma"] ** 2)
    return logr - _logsumexp(logr, axis=1)[:, None]

def _sticky_predict(bc, rho, K):
    return rho * bc + (1.0 - rho) * (1.0 - bc) / max(K - 1, 1)

def fit_condition_model(DESC, TRACKS):
    names = list(DESC.keys())
    if CONFIG.USE_TRAIN_CONDITION_LABELS and len(names) >= 2:
        C = np.stack([DESC[s].mean(0) for s in names]).astype(np.float32)
    else:
        pool = np.concatenate([DESC[s] for s in names], 0); K = max(2, len(names))
        rng = np.random.default_rng(CONFIG.SEED); C = pool[rng.choice(len(pool), K, replace=False)].copy()
        for _ in range(25):
            a = np.argmin(_pairwise_sq_dists(pool, C), 1)
            nC = np.stack([pool[a == k].mean(0) if (a == k).any() else C[k] for k in range(K)])
            if np.allclose(nC, C, atol=1e-5): C = nC; break
            C = nC
        names = [f"cond{k}" for k in range(K)]
    pool = np.concatenate([DESC[s] for s in DESC], 0)
    sigma = float(np.sqrt(np.median(_pairwise_sq_dists(pool, C).min(1))) + 1e-6)
    tau, season_list = [], list(DESC.keys())
    for i, s in enumerate(names):
        src = season_list[i] if s in DESC else season_list[i % len(season_list)]
        ref = ([o for o in season_list if o != src] or [src])[0]
        sims = cosine_sim(DESC[src], DESC[ref])
        gt = gt_match_index(TRACKS[src], TRACKS[ref]) if (src in TRACKS and ref in TRACKS) else np.arange(sims.shape[0])
        tr = (np.arange(sims.shape[0]) % 2 == 0) & (gt >= 0)
        if tr.sum() < 5: tr = (gt >= 0)
        tau.append(fit_temperature(sims[tr], gt[tr])[0])
    return dict(names=names, C=C, sigma=sigma, tau=np.asarray(tau, np.float32))

def run_ccbf(sim, query_desc, COND, null_quantile=0.98, exit_prob=0.01, return_traj=False):
    T, M = sim.shape; K = len(COND["tau"])
    inv_tau = (1.0 / np.maximum(COND["tau"], 1e-4)).astype(np.float64)
    appear = cond_log_resp(query_desc, COND); offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)
    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR; bc = np.full(K, 1.0 / K)
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    traj = np.empty((T, K)) if return_traj else None
    for t in range(T):
        bc = _sticky_predict(bc, CONFIG.STICKY_RHO, K)
        lbc = np.log(bc + 1e-30) + appear[t]; lbc -= _logsumexp(lbc); bc = np.exp(lbc)
        pp = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) + null * (1 - CONFIG.NULL_SELF) / M
        pn = CONFIG.NULL_SELF * null + exit_prob * place.sum()
        A = sim[t][None, :] * inv_tau[:, None] + np.log(bc + 1e-30)[:, None]
        ll_place = _logsumexp(A, axis=0)
        ll_null = _logsumexp(np.log(bc + 1e-30) + np.quantile(sim[t], null_quantile) * inv_tau)
        logb = np.empty(M + 1); logb[:M] = np.log(pp + 1e-30) + ll_place; logb[M] = np.log(pn + 1e-30) + ll_null
        logb -= _logsumexp(logb); b = np.exp(logb)
        place, null = b[:M], b[M]; s = place.sum() + null; place, null = place / s, null / s
        preds[t], confs[t], nulls[t] = int(np.argmax(place)), float(place.max()), float(null)
        if return_traj: traj[t] = bc
    return (preds, confs, nulls, traj) if return_traj else (preds, confs, nulls)

def gt_match_index(q, r):
    pos = {fid: j for j, fid in enumerate(r.frame_ids)}
    return np.array([pos.get(fid, -1) for fid in q.frame_ids], dtype=np.int64)

log("lib", "ok", "metrics + baselines + CCBF core loaded")

In [ ]:
# === L.4 — Reliability diagram helper ==================================================
def plot_reliability(curves, title, fname=None):
    """curves: dict name -> (conf, correct). Draws reliability diagram + ECE in legend."""
    try:
        import matplotlib.pyplot as plt
    except Exception as e:
        print("(plot skipped:", repr(e)[:60], ")"); return
    fig, ax = plt.subplots(figsize=(5.2, 5))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    for name, (conf, corr) in curves.items():
        ece, rows = ece_from_conf(conf, corr, CONFIG.ECE_BINS)
        if rows:
            xs = [r[1] for r in rows]; ys = [r[2] for r in rows]
            ax.plot(xs, ys, "o-", ms=4, label=f"{name} (ECE={ece:.3f})")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title(title)
    ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    if fname:
        out = os.path.join(CONFIG.WORKING_DIR, fname); fig.tight_layout(); fig.savefig(out, dpi=110); print("saved", out)
    plt.show()

## Data — Nordland (up to 4 synchronized seasons)

Seasons are row-aligned on the common frame-id grid, so row *i* is the same place under every season (clean GT).
With Nordland Full this gives **four genuine conditions** (spring/summer/fall/winter); with only Short it falls
back to summer/winter. Descriptors are cached (reused across NB1/NB2).

In [ ]:
# === D.1 — Track, transform, cached extraction =========================================
from PIL import Image
def _frame_id(f):
    d = "".join(c for c in os.path.splitext(f)[0] if c.isdigit()); return int(d) if d else None
def _imgs(d): return [f for f in os.listdir(d) if f.lower().endswith((".png", ".jpg", ".jpeg"))] if d and os.path.isdir(d) else []

class Track:
    def __init__(self, name, condition, image_dir):
        self.name, self.condition, self.image_dir = name, condition, image_dir
        pairs = sorted([(_frame_id(f), f) for f in _imgs(image_dir) if _frame_id(f) is not None])
        self.frame_ids = [a for a, _ in pairs]; self.paths = [os.path.join(image_dir, b) for _, b in pairs]
    def __len__(self): return len(self.paths)

try:
    import torchvision.transforms as T
    def build_transform(sz): return T.Compose([T.Resize((sz, sz), antialias=True), T.ToTensor(),
                                                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    _HAS_TV = True
except Exception:
    _HAS_TV = False
    def build_transform(sz):
        mean = np.array([0.485, 0.456, 0.406], np.float32); std = np.array([0.229, 0.224, 0.225], np.float32)
        def tx(im):
            a = (np.asarray(im.resize((sz, sz)).convert("RGB"), np.float32) / 255.0 - mean) / std
            return torch.from_numpy(a.transpose(2, 0, 1).copy())
        return tx
if not _HAS_TV: CONFIG.NUM_WORKERS = 0

class ImageList(torch.utils.data.Dataset):
    def __init__(self, paths, sz): self.paths = paths; self.tx = build_transform(sz); self.sz = sz
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try: im = Image.open(self.paths[i]).convert("RGB")
        except Exception: im = Image.new("RGB", (self.sz, self.sz))
        return self.tx(im), i

@torch.no_grad()
def extract_paths(paths, frame_ids, tag, force=False):
    cp = os.path.join(CONFIG.cache_dir, tag + ".npy"); mp = os.path.join(CONFIG.cache_dir, tag + ".json")
    if (not force) and os.path.isfile(cp) and os.path.isfile(mp):
        man = json.load(open(mp))
        if man.get("extractor") == EXTRACTOR_INFO["name"] and man.get("frame_ids") == list(frame_ids):
            print("  cache hit:", tag); return np.load(cp).astype(np.float32)
    dl = torch.utils.data.DataLoader(ImageList(paths, CONFIG.IMG_SIZE), batch_size=CONFIG.BATCH_SIZE,
                                     shuffle=False, num_workers=CONFIG.NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    feats = np.empty((len(paths), EXTRACTOR_INFO["dim"]), np.float32)
    amp = torch.bfloat16 if CONFIG.USE_BF16 else torch.float16; t0 = time.time()
    for imgs, idx in dl:
        imgs = imgs.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            with torch.autocast("cuda", dtype=amp): d = EXTRACT_FWD(imgs)
        else: d = EXTRACT_FWD(imgs)
        feats[idx.numpy()] = torch.nn.functional.normalize(d.float(), dim=1).cpu().numpy()
    np.save(cp, feats); json.dump({"extractor": EXTRACTOR_INFO["name"], "dim": int(EXTRACTOR_INFO["dim"]),
                                   "frame_ids": list(frame_ids)}, open(mp, "w"))
    assert np.isfinite(feats).all(), f"non-finite descriptors in {tag}"
    print(f"  extracted {tag}: {feats.shape} in {time.time()-t0:.1f}s"); return feats

In [ ]:
# === D.2 — Build aligned Nordland season tracks + descriptors ==========================
def _season_dir(root, season):
    cands = [c for c, _, fs in os.walk(root) if os.path.basename(c).lower() == season
             and any(f.lower().endswith((".png", ".jpg")) for f in fs)]
    return sorted(cands, key=lambda p: -p.count(os.sep))[0] if cands else None

# Prefer the 4-season FULL set (subsampled in smoke) so multi-condition experiments have real conditions;
# fall back to the 2-season Short set only if Full is not attached.
NORD_ROOT = PATHS["nordland_full"] or PATHS["nordland_short"]
ALL_SEASONS = ["spring", "summer", "fall", "winter"]
raw = {}
if NORD_ROOT:
    for s in ALL_SEASONS:
        d = _season_dir(NORD_ROOT, s)
        if d: raw[s] = Track(f"nordland_{s}", s, d)
# align on common frame-ids; subsample in smoke
TRACKS = {}
if len(raw) >= 2:
    common = sorted(set.intersection(*[set(t.frame_ids) for t in raw.values()]))
    if CONFIG.SMOKE_TEST and len(common) > CONFIG.SMOKE_MAX_FRAMES:
        sel = np.linspace(0, len(common) - 1, CONFIG.SMOKE_MAX_FRAMES).round().astype(int)
        common = [common[i] for i in sorted(set(sel.tolist()))]
    cset = set(common)
    for s, t in raw.items():
        keep = [(f, p) for f, p in zip(t.frame_ids, t.paths) if f in cset]
        t.frame_ids = [f for f, _ in keep]; t.paths = [p for _, p in keep]; TRACKS[s] = t
    log("D.2.tracks", "ok", f"seasons={list(TRACKS)} | aligned frames={len(common)}")
else:
    log("D.2.tracks", "fail", "need >=2 Nordland seasons")

DESC = {}
for s, t in TRACKS.items():
    DESC[s] = extract_paths(t.paths, t.frame_ids, tag=f"nordland_{'smoke' if CONFIG.SMOKE_TEST else 'full'}_{s}")
log("D.2.desc", "ok" if DESC else "fail", f"descriptors: {{{', '.join(f'{k}:{v.shape}' for k,v in DESC.items())}}}" if DESC else "none")

## NB7 — CRSG cross-dataset on Nordland (dense, frame-aligned GT)

Same reliability protocol as NB6 (RobotCar), now on Nordland's 4 seasons. Nordland is a **dense** traversal with
1:1 frame alignment — the sequential filter's natural home (it genuinely helped here in earlier work). So this is
the strong test: does per-frame **CRSG** match or beat the sequential filter even where the filter is best?

Place GT = frame-id identity (row-aligned). Correctness = |predicted ref index − GT index| ≤ tol (frames); we
sweep tol ∈ {0,1,2,5}. Ref map = summer; queries = the other seasons.

In [ ]:
# === N.0 — helpers (geometry features, logistic, risk-coverage) + episode ============
def geom_features(sim, k=5):
    part = np.sort(sim, 1)[:, ::-1][:, :max(k, 2)]
    maxs = part[:, 0]; gap12 = part[:, 0] - part[:, 1]; gap1k = part[:, 0] - part[:, 1:k].mean(1)
    p = np.exp(sim / 0.05 - _logsumexp(sim / 0.05, 1)[:, None]); ent = -(p * np.log(p + 1e-12)).sum(1)
    sue = np.sqrt(np.maximum(2 - 2 * maxs, 0.0))
    return np.stack([gap12, gap1k, maxs, sue, ent], 1), ["gap12", "gap1k", "maxsim", "sue", "entropy"]
def fit_logistic(X, y, iters=800, lr=0.3, l2=1e-4):
    mu = X.mean(0); sd = X.std(0) + 1e-9; Z = (X - mu) / sd; w = np.zeros(X.shape[1]); b = 0.0; n = len(Z)
    for _ in range(iters):
        p = 1 / (1 + np.exp(-(Z @ w + b))); w -= lr * (Z.T @ (p - y) / n + l2 * w); b -= lr * (p - y).mean()
    return lambda Xx: 1 / (1 + np.exp(-(((Xx - mu) / sd) @ w + b)))
def risk_coverage(conf, correct):
    o = np.argsort(-conf); err = 1.0 - correct[o]; return None, None, float((np.cumsum(err) / np.arange(1, len(err) + 1)).mean())
def softmax_conf(sim, tau):
    lp = sim / tau; pr = np.exp(lp - _logsumexp(lp, 1)[:, None]); return pr.max(1)

SEASONS = list(DESC.keys())
REF = "summer" if "summer" in SEASONS else SEASONS[0]
QS = [s for s in SEASONS if s != REF]
print("Nordland seasons:", SEASONS, "| ref =", REF, "| queries =", QS, "| frames/season:",
      {s: DESC[s].shape[0] for s in SEASONS})
RUN = len(SEASONS) >= 2
RES7 = OrderedDict()

def episode(season):
    sim = cosine_sim(DESC[season], DESC[REF]); gt = gt_match_index(TRACKS[season], TRACKS[REF])
    v = gt >= 0; sim = sim[v]; gt = gt[v]; pred = sim.argmax(1)
    return dict(sim=sim, gt=gt, qd=DESC[season][v], pred_err=np.abs(pred - gt))
log("N.0", "ok" if RUN else "fail", f"seasons={SEASONS} ref={REF}")

In [ ]:
# === N.1 — per-frame table (features + top-1 frame error) ============================
if RUN:
    X, pe, cond_id, sim_rows, qd_rows, gt_rows = [], [], [], [], [], []
    for ci, s in enumerate(QS):
        ep = episode(s); f, FN = geom_features(ep["sim"])
        X.append(f); pe.append(ep["pred_err"]); cond_id.append(np.full(len(ep["pred_err"]), ci))
        sim_rows.append(ep["sim"]); qd_rows.append(ep["qd"]); gt_rows.append(ep["gt"])
    TAB = dict(X=np.concatenate(X), pred_err=np.concatenate(pe), cond=np.concatenate(cond_id), FN=FN,
               sim=np.concatenate(sim_rows), qd=np.concatenate(qd_rows), gt=np.concatenate(gt_rows))
    gi = {n: i for i, n in enumerate(FN)}; TOLS = [0, 1, 2, 5]
    print(f"pooled N={len(TAB['pred_err'])} | R@tol (frames):")
    for t in TOLS:
        per = {QS[i]: round(float((TAB['pred_err'][TAB['cond'] == i] <= t).mean()), 2) for i in range(len(QS))}
        print(f"  tol={t:>2}  acc={float((TAB['pred_err'] <= t).mean()):.3f}  per-season={per}")
    log("N.1", "ok", f"N={len(TAB['pred_err'])}")

In [ ]:
# === N.2 — pooled cross-fit calibration swept over tolerance =========================
if RUN:
    rng = np.random.default_rng(CONFIG.SEED); fitm = rng.random(len(TAB["pred_err"])) < 0.5
    S = TAB["sim"][fitm]; G = TAB["gt"][fitm]; TAU_G = fit_temperature(S, G)[0]
    def eval_at_tol(tol):
        y = (TAB["pred_err"] <= tol).astype(float)
        Xf, yf, Xe, ye = TAB["X"][fitm], y[fitm], TAB["X"][~fitm], y[~fitm]; simE = TAB["sim"][~fitm]
        cols = [gi["gap12"], gi["sue"], gi["entropy"]]
        cals = {"softmax (uncal)": softmax_conf(simE, 0.05), "temperature": softmax_conf(simE, TAU_G),
                "platt(margin)": fit_logistic(Xf[:, [gi["gap12"]]], yf)(Xe[:, [gi["gap12"]]]),
                "CRSG (ours)": fit_logistic(Xf[:, cols], yf)(Xe[:, cols])}
        _, _, aopt = risk_coverage(ye + 1e-6 * np.random.default_rng(0).random(len(ye)), ye)
        rows = {m: {"ECE": float(ece_from_conf(c, ye, CONFIG.ECE_BINS)[0]), "AURC": float(risk_coverage(c, ye)[2]),
                    "conf_auroc": float(auroc(c, ye))} for m, c in cals.items()}
        return rows, float(aopt), float(ye.mean())
    RES7["sweep"] = {}
    for t in TOLS:
        rows, aopt, acc = eval_at_tol(t); RES7["sweep"][t] = {"rows": rows, "aurc_oracle": aopt, "acc": acc}
        print(f"\n[tol={t}  acc={acc:.3f}  oracleAURC={aopt:.4f}]  {'method':<16}{'ECE':>8}{'AURC':>8}{'conf-AUROC':>11}")
        for m in rows: print(f"{'':>34}{m:<16}{rows[m]['ECE']:>8.4f}{rows[m]['AURC']:>8.4f}{rows[m]['conf_auroc']:>11.4f}")
    TOL_HEAD = min(TOLS, key=lambda t: abs(RES7["sweep"][t]["acc"] - 0.80))
    RES7["MA"] = dict(RES7["sweep"][TOL_HEAD], tol=TOL_HEAD)
    print(f"\nheadline tol = {TOL_HEAD} frames (acc {RES7['MA']['acc']:.3f})")
    log("N.2", "ok", f"headline tol={TOL_HEAD} CRSG conf-AUROC={RES7['MA']['rows']['CRSG (ours)']['conf_auroc']:.3f}")

In [ ]:
# === N.3 — sequential filter & CCBF (Nordland = the filter's natural home) ===========
if RUN:
    th = RES7["MA"]["tol"]
    COND = fit_condition_model(DESC, TRACKS)
    seq = {"fixed_filter": [[], []], "ccbf": [[], []]}
    for s in QS:
        ep = episode(s)
        pf, cf, _ = run_place_filter(ep["sim"], TAU_G); seq["fixed_filter"][0].append(cf); seq["fixed_filter"][1].append((np.abs(pf - ep["gt"]) <= th).astype(float))
        pc, cc, _ = run_ccbf(ep["sim"], ep["qd"], COND); seq["ccbf"][0].append(cc); seq["ccbf"][1].append((np.abs(pc - ep["gt"]) <= th).astype(float))
    print(f"\nsequential baselines (tol={th} frames):  {'method':<14}{'ECE':>8}{'AURC':>8}{'conf-AUROC':>11}")
    for name in seq:
        conf = np.concatenate(seq[name][0]); corr = np.concatenate(seq[name][1])
        RES7["MA"]["rows"][name] = {"ECE": float(ece_from_conf(conf, corr, CONFIG.ECE_BINS)[0]), "AURC": float(risk_coverage(conf, corr)[2]), "conf_auroc": float(auroc(conf, corr))}
        print(f"{'':>30}{name:<14}{RES7['MA']['rows'][name]['ECE']:>8.4f}{RES7['MA']['rows'][name]['AURC']:>8.4f}{RES7['MA']['rows'][name]['conf_auroc']:>11.4f}  (acc={corr.mean():.3f})")
    log("N.3", "ok", "filter+ccbf compared")

In [ ]:
# === N.4 — off-map + verdict =========================================================
if RUN:
    _yh = (TAB["pred_err"] <= RES7["MA"]["tol"]).astype(float)
    cols = [gi["gap12"], gi["sue"], gi["entropy"]]; CRSG_CAL = fit_logistic(TAB["X"][fitm][:, cols], _yh[fitm])
    frac = CONFIG.OFFMAP_MAP_FRACTION; M = DESC[REF].shape[0]; Mmap = int(M * frac)
    ag = {m: {"s": [], "l": []} for m in ["sue", "filter_null", "crsg_unmap"]}
    for s in QS:
        ep = episode(s); lab = (ep["gt"] >= Mmap).astype(int)
        if lab.sum() == 0 or lab.sum() == len(lab): continue
        sim2 = cosine_sim(ep["qd"], DESC[REF][:Mmap]); f2, _ = geom_features(sim2); _, _, nf = run_place_filter(sim2, TAU_G)
        rel = CRSG_CAL(f2[:, cols])
        for m, sc in [("sue", f2[:, gi["sue"]]), ("filter_null", nf), ("crsg_unmap", 1 - rel)]:
            ag[m]["s"].append(sc); ag[m]["l"].append(lab)
    off = {m: {"AUROC": auroc(np.concatenate(ag[m]["s"]), np.concatenate(ag[m]["l"]))} for m in ag if ag[m]["s"]}
    print("\noff-map detection:", {m: round(off[m]["AUROC"], 3) for m in off})
    RH = RES7["MA"]["rows"]; ours = RH["CRSG (ours)"]
    base_aurc = min(RH[m]["AURC"] for m in ["temperature", "fixed_filter", "ccbf"] if m in RH)
    crsg_ok = (ours["conf_auroc"] >= max(RH[m]["conf_auroc"] for m in RH) - 1e-9) and (ours["AURC"] <= base_aurc + 1e-9)
    summary = dict(dataset="nordland", ref=REF, seasons=SEASONS, headline_tol=RES7["MA"]["tol"], acc=round(RES7["MA"]["acc"], 3),
                   headline={m: {"ECE": round(RH[m]["ECE"], 4), "AURC": round(RH[m]["AURC"], 4), "conf_auroc": round(RH[m]["conf_auroc"], 4)} for m in RH},
                   offmap={m: round(off[m]["AUROC"], 4) for m in off},
                   sweep={str(t): {"acc": round(RES7["sweep"][t]["acc"], 3), "crsg_aurc": round(RES7["sweep"][t]["rows"]["CRSG (ours)"]["AURC"], 4),
                                   "crsg_conf_auroc": round(RES7["sweep"][t]["rows"]["CRSG (ours)"]["conf_auroc"], 4)} for t in TOLS},
                   crsg_best=bool(crsg_ok))
    print("\n================ NORDLAND CRSG SUMMARY (headline tol={} frames) ================".format(RES7["MA"]["tol"]))
    for m in RH: print(f"  {m:<16} ECE={RH[m]['ECE']:.4f} AURC={RH[m]['AURC']:.4f} conf-AUROC={RH[m]['conf_auroc']:.4f}")
    print("NORD_JSON=" + json.dumps(summary)); log("N.4", "ok", f"crsg_best={crsg_ok}")
else:
    print("Nordland not available (need >=2 seasons). seasons found:", SEASONS)
    log("N.4", "fail", "nordland not attached / <2 seasons")